# PyTorch Neural Network para Classificação Binária

MLP com `torch.nn` para classificar entre **duas classes** (0 ou 1).

### Componentes principais
| Componente | Escolha | Por quê |
|---|---|---|
| Última ativação | `nn.Sigmoid()` | Comprime saída para [0,1] → probabilidade |
| Loss | `nn.BCELoss` | Binary Cross Entropy |
| Otimizador | `Adam` | Adaptativo, robusto |
| Threshold | `0.5` | Converter prob → classe |

## 1. Importar Bibliotecas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, roc_auc_score)
from sklearn.datasets import make_classification

# Reprodutibilidade
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch versão: {torch.__version__}")

## 2. Gerar e Preparar Dados

Usamos `make_classification` do scikit-learn para gerar um problema de classificação binária
com 10 features (5 informativas, 2 redundantes) — dados tabulares típicos.

A normalização é fundamental para que os gradientes tenham magnitudes compatíveis.

In [ ]:
# Dados de classificação binária (2 classes)
X, y = make_classification(
    n_samples=400, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2, random_state=42
)

# Divisão treino / teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalização
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Converter para tensores
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train).reshape(-1, 1)   # shape [N, 1] para BCELoss
X_test  = torch.FloatTensor(X_test)
y_test  = torch.FloatTensor(y_test).reshape(-1, 1)

print(f"Treino : X={X_train.shape}, y={y_train.shape}")
print(f"Teste  : X={X_test.shape},  y={y_test.shape}")
print(f"Dist. classes (treino): {y_train.sum().int().item()} pos / {(1-y_train).sum().int().item()} neg")

## 3. Arquitetura da Rede Neural

**Diferença-chave em relação à regressão:**

A **última camada** usa `nn.Sigmoid()` que comprime a saída para o intervalo **[0, 1]**,
interpretável como probabilidade de pertencer à classe positiva.

```
x₁ = ReLU(W₀·x  + b₀)
x₂ = ReLU(W₁·x₁ + b₁)
ŷ  = σ(W₂·x₂ + b₂)      ← σ = Sigmoid → probabilidade ∈ (0, 1)
```

> *"Sigmoid compresses the output to the (0,1) range — the natural range for probabilities."*

In [ ]:
class BinaryClassifierNN(nn.Module):
    """
    MLP para Classificação Binária
    Arquitetura: 10 → 64 → 32 → 1 + Sigmoid
    A Sigmoid na saída produz P(classe=1) ∈ (0,1)
    """
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),                       # ativação interna — não-linearidade
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()                     # saída: probabilidade ∈ (0,1)
        )

    def forward(self, x):
        return self.network(x)

model = BinaryClassifierNN(input_size=10, hidden_size=64)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal de parâmetros treináveis: {n_params:,}")

## 4. Loss e Otimizador

- **`nn.BCELoss`**: Binary Cross Entropy — mede a divergência entre a distribuição prevista
  (probabilidades do Sigmoid) e os rótulos reais {0, 1}. Penaliza fortemente predições
  confiantes e erradas.
- **`Adam`**: otimizador adaptativo.

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Loss function : {criterion}")
print(f"Otimizador    : Adam  (lr=0.001)")

## 5. Ciclo de Treino com Acompanhamento de Validação

Padrão padrão:
- **`model.train()`** antes de cada batch de treino
- **`model.eval()` + `torch.no_grad()`** para a passagem de validação
- Monitoramos **loss** e **acurácia** de treino e validação a cada época

In [ ]:
epochs       = 150
train_losses = []
val_losses   = []
train_accs   = []
val_accs     = []

for epoch in range(epochs):

    # ── Fase de Treino ────────────────────────────────────────────────
    model.train()
    outputs = model(X_train)
    loss    = criterion(outputs, y_train)

    optimizer.zero_grad()                # zera gradientes acumulados
    loss.backward()                      # autograd — retropropagação
    optimizer.step()

    # Acurácia de treino
    preds_train = (outputs.detach() > 0.5).float()
    acc_train   = (preds_train == y_train).float().mean().item()
    train_losses.append(loss.item())
    train_accs.append(acc_train)

    # ── Fase de Validação — sem gradientes ────────────────────────────
    model.eval()
    with torch.no_grad():
        val_out  = model(X_test)
        val_loss = criterion(val_out, y_test)
        preds_val = (val_out > 0.5).float()
        acc_val   = (preds_val == y_test).float().mean().item()

    val_losses.append(val_loss.item())
    val_accs.append(acc_val)

    if (epoch + 1) % 30 == 0:
        print(f"Epoch [{epoch+1:3d}/{epochs}] | "
              f"Train Loss: {loss.item():.4f}  Acc: {acc_train:.4f} | "
              f"Val Loss: {val_loss.item():.4f}  Acc: {acc_val:.4f}")

print("\nTreinamento concluído!")

## 6. Avaliação e Visualização

Avaliamos com métricas de classificação binária:
- **Acurácia**: percentual de acertos
- **Precisão** / **Recall**: relevante quando classes são desbalanceadas
- **F1-Score**: média harmônica de precisão e recall
- **AUC-ROC**: área sob a curva ROC (1.0 = perfeito)

Verificamos também as curvas de treino/validação para detectar **overfitting**.

In [ ]:
model.eval()
with torch.no_grad():
    probs_train = model(X_train).numpy()
    probs_test  = model(X_test).numpy()

y_pred_train = (probs_train > 0.5).astype(int)
y_pred_test  = (probs_test  > 0.5).astype(int)

# Métricas
acc_tr  = accuracy_score(y_train.numpy(), y_pred_train)
acc_te  = accuracy_score(y_test.numpy(),  y_pred_test)
prec_te = precision_score(y_test.numpy(), y_pred_test)
rec_te  = recall_score(y_test.numpy(),    y_pred_test)
f1_te   = f1_score(y_test.numpy(),        y_pred_test)
auc_te  = roc_auc_score(y_test.numpy(),   probs_test)

print("=== Métricas de Desempenho ===")
print(f"Acurácia  Treino : {acc_tr:.4f}")
print(f"Acurácia  Teste  : {acc_te:.4f}")
print(f"Precisão  Teste  : {prec_te:.4f}")
print(f"Recall    Teste  : {rec_te:.4f}")
print(f"F1-Score  Teste  : {f1_te:.4f}")
print(f"AUC-ROC   Teste  : {auc_te:.4f}")

In [ ]:
cm = confusion_matrix(y_test.numpy(), y_pred_test)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Plot 1: Curvas de Loss (Cap. 5.5.3 — Figura 5.14) ────────────────────────
axes[0].plot(train_losses, label='Treino',    color='royalblue')
axes[0].plot(val_losses,   label='Validação', color='darkorange', linestyle='--')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss (BCE)')
axes[0].set_title('Curvas de Loss\n(Cap. 5.5.3 — Overfitting?)')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# ── Plot 2: Curvas de Acurácia ────────────────────────────────────────────────
axes[1].plot(train_accs, label='Treino',    color='royalblue')
axes[1].plot(val_accs,   label='Validação', color='darkorange', linestyle='--')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia')
axes[1].set_title('Curvas de Acurácia\nTreino vs Validação')
axes[1].legend()
axes[1].grid(True, alpha=0.4)
axes[1].set_ylim(0, 1.05)

# ── Plot 3: Matriz de Confusão ────────────────────────────────────────────────
im = axes[2].imshow(cm, cmap='Blues', interpolation='nearest')
plt.colorbar(im, ax=axes[2])
axes[2].set_title(f'Matriz de Confusão\nAcurácia Teste: {acc_te:.4f}')
axes[2].set_xlabel('Predição')
axes[2].set_ylabel('Real')
axes[2].set_xticks([0, 1]); axes[2].set_yticks([0, 1])
axes[2].set_xticklabels(['Neg (0)', 'Pos (1)'])
axes[2].set_yticklabels(['Neg (0)', 'Pos (1)'])
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)

plt.tight_layout()
plt.savefig('binaria_resultados.png', dpi=120, bbox_inches='tight')
plt.show()